In [2]:
import sqlite3
con = sqlite3.connect("practice.db")
con.executescript(open("practice_setup.sql").read())
import pandas as pd
pd.read_sql("SELECT * FROM employees", con) 

,emp_id,name,dept_id,salary,hire_date,manager_id
0,1,Arjun,1,95000,2019-03-15,NaN
1,2,Priya,1,72000,2020-07-01,1.0
2,3,Karan,1,68000,2021-01-20,1.0
3,4,Divya,1,81000,2020-11-11,1.0
4,5,Rohit,2,60000,2021-05-30,6.0
5,6,Sneha,2,88000,2018-09-10,NaN
6,7,Vikram,2,55000,2022-02-14,6.0
7,8,Anita,3,50000,2021-08-19,NaN
8,9,Manoj,3,47000,2022-06-25,8.0
9,10,Kavya,4,76000,2019-12-01,NaN


In [4]:
"""-- 1. Basic Join  [Service / joins]
-- List every employee's name alongside their department name."""
query1 = """Select e.name, d.dept_name from employees e JOIN departments d on d.dept_id=e.dept_id"""
pd.read_sql(query1,con)

,name,dept_name
0,Arjun,Engineering
1,Priya,Engineering
2,Karan,Engineering
3,Divya,Engineering
4,Rohit,Sales
5,Sneha,Sales
6,Vikram,Sales
7,Anita,HR
8,Manoj,HR
9,Kavya,Finance


In [5]:
"""-- 2. INNER vs LEFT  [AI screen / joins]
-- List ALL employees with their department name, including Nisha
-- (who is in a department that doesn't exist). She should appear
-- with a NULL / None department.
-- (which join keeps her?)"""
query2 = """Select e.name, d.dept_name from employees e LEFT JOIN departments d on d.dept_id=e.dept_id"""
pd.read_sql(query2,con)

,name,dept_name
0,Arjun,Engineering
1,Priya,Engineering
2,Karan,Engineering
3,Divya,Engineering
4,Rohit,Sales
5,Sneha,Sales
6,Vikram,Sales
7,Anita,HR
8,Manoj,HR
9,Kavya,Finance


In [7]:
"""-- 3. Join With a Filter  [Service / joins]
-- List the names of all employees who work in Chennai.
-- (join employees to departments, then filter on location)"""
query3 = """Select e.name, d.dept_name,d.location from employees e JOIN departments d on d.dept_id=e.dept_id WHERE d.location ='Chennai'"""
pd.read_sql(query3,con)

,name,dept_name,location
0,Arjun,Engineering,Chennai
1,Priya,Engineering,Chennai
2,Karan,Engineering,Chennai
3,Divya,Engineering,Chennai
4,Anita,HR,Chennai
5,Manoj,HR,Chennai
6,Ramesh,Engineering,Chennai


In [10]:
"""-- 4. Join Three Tables  [AI screen / joins]
-- List each employee name and the project name they are assigned to.
-- (employees -> assignments -> projects)"""
query3 = """Select e.name,p.project_name from assignments a JOIN employees e on e.emp_id = a.emp_id JOIN projects p on p.project_id = a.project_id"""
pd.read_sql(query3,con)

,name,project_name
0,Arjun,Payment Gateway
1,Priya,Payment Gateway
2,Priya,Mobile App
3,Karan,Payment Gateway
4,Divya,Mobile App
5,Rohit,CRM Rollout
6,Vikram,CRM Rollout
7,Kavya,Audit System
8,Suresh,Audit System
9,Deepa,Q4 Campaign


In [15]:
"""-- 5. Self Join  [AI screen / joins]
-- List each employee's name alongside their manager's name.
-- Hint: join the employees table to itself. One alias for the
-- employee, another alias for the manager, matching
-- employee.manager_id = manager.emp_id."""
query4 = """Select e.name,em.name as 'Manager_name' from employees e
JOIN employees em on e.manager_id = em.emp_id """
pd.read_sql(query4,con)

,name,Manager_name
0,Priya,Arjun
1,Karan,Arjun
2,Divya,Arjun
3,Rohit,Sneha
4,Vikram,Sneha
5,Manoj,Anita
6,Suresh,Kavya
7,Ramesh,Arjun
8,Lakshmi,Sneha


In [17]:
"""-- 1. Count Per Group  [Service / group by]
-- How many employees are in each department? (use dept_id)"""
query5 = """Select d.dept_name,count(e.name) as 'Emp_Count' from employees e
JOIN departments d on e.dept_id = d.dept_id 
group by e.dept_id"""
pd.read_sql(query5,con)

,dept_name,Emp_Count
0,Engineering,5
1,Sales,4
2,HR,2
3,Finance,2
4,Marketing,1


In [25]:
"""-- 2. Average Per Group  [AI screen / group by]
-- What is the average salary in each department? (use dept_id)"""
query6 = """Select e.name,AVG(e.salary) as 'AVG_Salary' from employees e
JOIN departments d on e.dept_id = d.dept_id 
group by e.dept_id"""
pd.read_sql(query6,con)

,name,AVG_Salary
0,Arjun,77200.0
1,Rohit,66250.0
2,Anita,48500.0
3,Kavya,70000.0
4,Deepa,59000.0


In [20]:
"""-- 3. Total With a Join  [AI screen / group by]
-- Total sales amount per salesperson, showing the employee's NAME.
-- (join sales to employees, group by the person, sum the amount)"""
query7 = """Select e.name,SUM(s.amount) as 'Total_Sales' from sales s
JOIN employees e on e.emp_id = s.emp_id 
group by e.emp_id"""
pd.read_sql(query7,con)

,name,Total_Sales
0,Rohit,65000
1,Sneha,77000
2,Vikram,33000
3,Lakshmi,25000


In [22]:
"""-- 4. HAVING  [AI screen / group by]
-- Show the dept_id and headcount for departments that have MORE THAN
-- 2 employees. (group, then filter groups with HAVING)"""
query7 = """Select d.dept_name,count(e.emp_id) as 'No_of_emp' from departments d
JOIN employees e on e.dept_id = d.dept_id 
group by e.dept_id
having No_of_emp>2"""
pd.read_sql(query7,con)

,dept_name,No_of_emp
0,Engineering,5
1,Sales,4


In [23]:
"""-- 5. WHERE + GROUP BY together  [AI screen / group by]
-- Among employees hired in 2021 or later, how many are in each department?
-- (filter rows first with WHERE on hire_date, THEN group)
-- Hint: hire_date is text like '2021-05-30', so hire_date >= '2021-01-01' works."""
query7 = """Select d.dept_name,count(e.emp_id) as 'No_of_emp' from departments d
JOIN employees e on e.dept_id = d.dept_id 
where e.hire_date >= '2021-01-01'
group by e.dept_id"""
pd.read_sql(query7,con)

,dept_name,No_of_emp
0,Engineering,2
1,Sales,3
2,HR,2
3,Finance,1
4,Marketing,1


In [26]:
"""-- 1. Above Average  [AI screen / subquery]
-- List the names and salaries of employees who earn MORE than the
-- company-wide average salary.
-- (scalar subquery in the WHERE)"""
query8 = """select name,salary from employees where salary > (select AVG(salary) from employees)  """
pd.read_sql(query8,con)

,name,salary
0,Arjun,95000
1,Priya,72000
2,Karan,68000
3,Divya,81000
4,Sneha,88000
5,Kavya,76000
6,Ramesh,70000


In [27]:
"""-- 2. IN with a Subquery  [AI screen / subquery]
-- List the names of employees who work in a department located in Chennai.
-- Use a subquery with IN (do NOT join this time).
-- (inner query returns the Chennai dept_ids)
"""
query9 = """select name,dept_id from employees where dept_id in (select dept_id from departments where location='Chennai') """
pd.read_sql(query9,con)

,name,dept_id
0,Arjun,1
1,Priya,1
2,Karan,1
3,Divya,1
4,Anita,3
5,Manoj,3
6,Ramesh,1


In [28]:
"""-- 3. Subquery for a Single Lookup  [Service / subquery]
-- List the names of employees who earn more than the employee named 'Priya'.
-- (inner query looks up Priya's salary, outer compares to it)"""
query10 = """select name,salary from employees where salary > (select salary from employees where name='Priya') """
pd.read_sql(query10,con)

,name,salary
0,Arjun,95000
1,Divya,81000
2,Sneha,88000
3,Kavya,76000
